In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev_lab.silver.reading_events (
    event_id STRING,
    user_id STRING,
    book_id STRING,
    event_time TIMESTAMP,
    duration_seconds INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO dev_lab.bronze.reading_events
VALUES
  ('evt301', 'user10', 'book70',
   TIMESTAMP '2026-10-18 12:00:00', 300),
  ('evt302', 'user11', 'book80',
   TIMESTAMP '2026-10-18 12:01:00', -50),
  ('evt301', 'user10', 'book70',
   TIMESTAMP '2026-10-18 12:00:00', 300);

In [0]:
%sql
INSERT INTO dev_lab.silver.reading_events
SELECT
    event_id,
    user_id,
    book_id,
    event_time,
    duration_seconds
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY event_id
            ORDER BY event_time DESC
        ) AS rn
    FROM dev_lab.bronze.reading_events
    WHERE event_id IS NOT NULL
      AND user_id IS NOT NULL
      AND book_id IS NOT NULL
      AND event_time IS NOT NULL
      AND duration_seconds >= 0
) valid
WHERE rn = 1;

In [0]:
%sql
SELECT *
FROM dev_lab.silver.reading_events
WHERE event_id IN ('evt301', 'evt302')
ORDER BY event_id;